In [11]:
import gc
import os
import csv
from speechbrain.inference.ASR import EncoderASR
from utils.read_transcription import *
from utils.normalise_text import *
from pathlib import Path
from hyperpyyaml import load_hyperpyyaml
import librosa
import torch
from utils.meta import get_audio_info
from utils.apply_vad import *
from utils.list_files import list_files
from utils.VAD_chunk import *
from utils.wer_chunk import wer_chunk
from utils.wer_segment import wer_segment


In [13]:
w2v = EncoderASR.from_hparams(source="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-wav2vec2-commonvoice-fr", savedir="/vol/experiments3/imbenamor/TAPAS-FRAIS/models/asr-wav2vec2-commonvoice-fr", run_opts={"device":"cuda"})
#wav_data="/vol/corpora/Rhapsodie/wav16k_corrected/"
#ref_trans= "/vol/corpora/Rhapsodie/TextGrids-fev2013/"
#wav_data = "/vol/corpora/TAPAS_FRAIS/Data_Partagees_Mons/Data_Mons/Description"
#ref_trans = "/vol/corpora/TAPAS_FRAIS/Data_Partagees_Mons/Mons TextGrid verifie + 50 ans"
#wav_data = "/vol/corpora/Rhapsodie/wav16k_corrected_withsilence"
#ref_trans = "/vol/corpora/Rhapsodie/TextGrid.phones.sampa"
#wav_data = "/vol/corpora/Rhapsodie/wav16k_corrected_withsilence"
#ref_trans = "/vol/corpora/Rhapsodie/TextGrid.phones.sampa"
ref_trans = "/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-CEREB"
wav_data = "/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-CEREB"

csv_path = "./w2v-trans_pyannote_cereb.csv"


speechbrain.lobes.models.huggingface_transformers.huggingface - Wav2Vec2Model is frozen.


In [14]:
def list_files(trans_dir):
    tg_to_wav = {}
    for f in sorted(os.listdir(trans_dir)):
        for w in os.listdir(wav_data):            
            if f.split(".")[0]==w.split(".")[0] and f.endswith("TextGrid") and w.endswith(".wav") and not w.startswith(".") and not f.endswith("pr_analyse.TextGrid"):
                tg_to_wav[f] = w
           

    return tg_to_wav
tg_to_wav = list_files(ref_trans)
tg_to_wav

{'CCM-002710-01_L01.TextGrid': 'CCM-002710-01_L01.wav',
 'CCM-003094-01_L01.TextGrid': 'CCM-003094-01_L01.wav',
 'CCM-003110-01_L01.TextGrid': 'CCM-003110-01_L01.wav',
 'CCM-003493-01_L01.TextGrid': 'CCM-003493-01_L01.wav',
 'CCM-003998-01_L01.TextGrid': 'CCM-003998-01_L01.wav',
 'CCM-004523-01_L01.TextGrid': 'CCM-004523-01_L01.wav',
 'CCM-004538-01_L01.TextGrid': 'CCM-004538-01_L01.wav',
 'CCM-004773-01_L01.TextGrid': 'CCM-004773-01_L01.wav'}

In [15]:
"""#montpage
def list_files(trans_dir):
    tg_to_wav = {}
    for f in sorted(os.listdir(trans_dir)):
        for w in os.listdir(wav_data):            
            if f.split(".")[0]+".wav" in w:
                tg_to_wav[f] = w
           

    return tg_to_wav
tg_to_wav = list_files(ref_trans)"""

In [5]:
def list_files(trans_dir):
    tg_to_wav = {}
    for f in sorted(os.listdir(trans_dir)):
        for w in os.listdir(wav_data):            
            if f.split("-")[1].split(".")[0] in w:
                tg_to_wav[f] = w
           

    return tg_to_wav
tg_to_wav = list_files(ref_trans)
tg_to_wav

{'Rhap-D0002-Pro.TextGrid': 'Rhap-D0002.wav',
 'Rhap-D0003-Pro.TextGrid': 'Rhap-D0003.wav',
 'Rhap-D0004-Pro.TextGrid': 'Rhap-D0004.wav',
 'Rhap-D0005-Pro.TextGrid': 'Rhap-D0005.wav',
 'Rhap-D0006-Pro.TextGrid': 'Rhap-D0006.wav',
 'Rhap-D0007-Pro.TextGrid': 'Rhap-D0007.wav',
 'Rhap-D0008-Pro.TextGrid': 'Rhap-D0008.wav',
 'Rhap-D0009-Pro.TextGrid': 'Rhap-D0009.wav',
 'Rhap-D0017-Pro.TextGrid': 'Rhap-D0017.wav',
 'Rhap-D0020-Pro.TextGrid': 'Rhap-D0020.wav',
 'Rhap-D1001-Pro.TextGrid': 'Rhap-D1001.wav',
 'Rhap-D1002-Pro.TextGrid': 'Rhap-D1002.wav',
 'Rhap-D2001-Pro.TextGrid': 'Rhap-D2001.wav',
 'Rhap-D2002-Pro.TextGrid': 'Rhap-D2002.wav',
 'Rhap-D2003-Pro.TextGrid': '._Rhap-D2003.wav',
 'Rhap-D2004-Pro.TextGrid': 'Rhap-D2004.wav',
 'Rhap-D2005-Pro.TextGrid': 'Rhap-D2005.wav',
 'Rhap-D2006-Pro.TextGrid': 'Rhap-D2006.wav',
 'Rhap-D2007-Pro.TextGrid': 'Rhap-D2007.wav',
 'Rhap-D2008-Pro.TextGrid': 'Rhap-D2008.wav',
 'Rhap-D2009-Pro.TextGrid': 'Rhap-D2009.wav',
 'Rhap-D2010-Pro.TextGrid': 'Rha

In [15]:
"""
WhisperX-style VAD chunking for the Whisper-encoder + CTC phoneme model.

Built on your approach: uses whisperx.vads.pyannote.load_vad_model, reads the
RAW per-frame VAD scores, and binarizes them with WhisperX's onset/offset
thresholds. The goal is unchanged from the original vad_chunk_with_timestamps:
return a list of {"start", "end"} (seconds, original time) chunks, each <= 30 s,
ready for the inference loop.

Two stages:
  1. VAD + binarize  -> WhisperX raw scores -> speech segments (hysteresis: go
                        active above `onset`, inactive below `offset`).
  2. cut & merge     -> pack segments into <= chunk_size windows, KEEPING internal
                        pauses inside a window. A single segment longer than
                        chunk_size is split at its QUIETEST frame (real WhisperX
                        behaviour, possible here because we kept the raw scores),
                        so nothing ever hits Whisper's 30 s truncation.

Only load_whisperx_vad() touches whisperx, so the rest is importable/testable
offline without it.
"""

import numpy as np
import torch
from whisperx.vads.pyannote import load_vad_model
# WhisperX defaults
ONSET = 0.5
OFFSET = 0.363


def load_whisperx_vad(wav):
    vad_pipeline = load_vad_model(
    device="cuda",
    token=os.environ["HF_TOKEN"])
    vad_scores = vad_pipeline(wav)
    scores = vad_scores.data[:, 0]
    frames = vad_scores.sliding_window
    times = [frames[i].middle for i in range(len(scores))]
    return scores, times

def _binarize(scores, times, onset=ONSET, offset=OFFSET):
    """Hysteresis binarization -> list of (start_s, end_s) speech segments."""
    segments = []
    is_active = scores[0] > onset
    start = times[0] if is_active else None
    for t, sc in zip(times[1:], scores[1:]):
        if is_active:
            if sc < offset:
                segments.append((start, t))
                is_active = False
        else:
            if sc > onset:
                start = t
                is_active = True
    if is_active:
        segments.append((start, times[-1]))
    return segments


def _split_long_by_score(seg_start, seg_end, times, scores, chunk_size):
    """Split a > chunk_size segment at the quietest frame in each window.

    Mirrors WhisperX: when a segment exceeds max_duration, cut at the lowest
    detection score in the second half rather than at a hard time boundary, so
    the cut lands on minimally-active speech.
    """
    pieces, cur = [], seg_start
    while seg_end - cur > chunk_size:
        lo, hi = cur + chunk_size * 0.5, cur + chunk_size
        idx = np.where((times >= lo) & (times <= hi))[0]
        cut = times[idx[np.argmin(scores[idx])]] if len(idx) else cur + chunk_size
        pieces.append((cur, cut))
        cur = cut
    pieces.append((cur, seg_end))
    return pieces


def merge_chunks(segments, times, scores, chunk_size=20.0):
    """WhisperX cut & merge -> list of {"start","end"} chunks, each <= chunk_size."""
    times = np.asarray(times, dtype=float)
    scores = np.asarray(scores, dtype=float)
    split = []
    for s, e in segments:
        if e - s > chunk_size:
            split.extend(_split_long_by_score(s, e, times, scores, chunk_size))
        else:
            split.append((s, e))

    if not split:
        return []

    merged = []
    curr_start, curr_end = split[0]
    for s, e in split[1:]:
        if e - curr_start > chunk_size and curr_end - curr_start > 0:
            merged.append({"start": curr_start, "end": curr_end})
            curr_start = s
        curr_end = e
    print(curr_end-curr_start)
    merged.append({"start": curr_start, "end": curr_end})
    return merged


def vad_chunk_with_timestamps(
    wav,
    sampling_rate=16000,
    max_chunk_duration=8.0,
    onset=ONSET,
    offset=OFFSET,
):
    """Drop-in replacement. Same return type as the old rVAD version.

    wav               : torch.Tensor (1D, 16 kHz)  -- or a file path
    vad_model         : object from load_whisperx_vad() (load it once, reuse it)
    max_chunk_duration: keep <= 30.0 for Whisper's hard cap
    """
    scores, times = load_whisperx_vad(wav)
    segments = _binarize(scores, times, onset, offset)
    return merge_chunks(segments, times, scores, chunk_size=max_chunk_duration)

In [16]:
def wer_segment(wav_file,ref_transcriptions,pred_transcriptions):
    wer_hparams = load_hyperpyyaml("""wer_stats: !new:speechbrain.utils.metric_stats.ErrorRateStats""")
    wer_hparams["wer_stats"].clear()
    wer_hparams["wer_stats"].append(ids=list(range(len(ref_transcriptions))),
                                    predict=[normalization(pred_transcriptions)],
                                    target=[normalization(ref_transcriptions)])
    stats = wer_hparams["wer_stats"].summarize()
    logger.info("File: %s | WER=%f | S=%d D=%d I=%d N=%d", wav_file.split("/")[-1], stats["WER"], stats["substitutions"],
                stats["deletions"], stats["insertions"],stats["num_scored_tokens"])
    logger.info("-" * 30)
    return stats["WER"], stats["substitutions"], stats["deletions"], stats["insertions"],stats["num_scored_tokens"]
def wer_chunk(results, words):

    wer_hparams = load_hyperpyyaml(
        """wer_stats: !new:speechbrain.utils.metric_stats.ErrorRateStats"""
    )

    for r in results:
        r["ref"] = remove_words(
            ref_text_for_chunk(words, r["start"], r["end"])
        )



    valid_chunks = [r for r in results if r["ref"].strip()]

    ref_full = " ".join(r["ref"] for r in valid_chunks)
    hyp_full = " ".join(r["text"] for r in valid_chunks)

    wer_hparams["wer_stats"].clear()

    wer_hparams["wer_stats"].append(
        ids=["file"],
        predict=[normalization(hyp_full)],
        target=[normalization(ref_full)]
    )

    file_stats = wer_hparams["wer_stats"].summarize()

    return ref_full, hyp_full, file_stats["WER"]

In [8]:
import pandas as pd
style = pd.read_csv("/vol/corpora/Rhapsodie/wav_style.csv")
style["wav_file"]="/vol/corpora/Rhapsodie/wav16k_corrected/Rhap-"+style["file"]+".wav"
wav_to_style = dict(
    zip(style["wav_file"], style["style"])
)
from collections import defaultdict

style_wers = defaultdict(
    lambda: load_hyperpyyaml(
        """wer_stats: !new:speechbrain.utils.metric_stats.ErrorRateStats"""
    )
)

In [17]:
corpus_wer = load_hyperpyyaml(
    """wer_stats: !new:speechbrain.utils.metric_stats.ErrorRateStats"""
)

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    print("===================================")
    fieldnames = ["filename", "duration_sec", "samplerate", "channels","trans_w2vec_vad_chunk","WER_w2vec_vad_chunk"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    wer_hparams = load_hyperpyyaml("""wer_stats: !new:speechbrain.utils.metric_stats.ErrorRateStats""")

    for i,(tg,wav) in enumerate(tg_to_wav.items()):
        wav_file = os.path.join(wav_data, wav)
        trans_file = os.path.join(ref_trans, tg)
        print(wav_file)
        if os.path.exists(wav_file) and os.path.exists(trans_file) and wav_file !="/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-CEREB/CCM-004773-01_L01.wav" and '002710' not in wav_file:
            info = get_audio_info(wav_file)
            # load audio
            audio_np, sr = read_audio_16k(wav_file)
            wav = torch.from_numpy(audio_np)
            # VAD + chunking
            chunks = vad_chunk_with_timestamps(wav_file,max_chunk_duration=30)
            words = get_textgrid_transcription_chunk(trans_file)
            #=========W2VEC vad chunk=========
            results = whisper_transcribe_chunks(w2v, "wav2vec2-VAD-chunk", wav, chunks)
            ref_transcriptions, pred_transcriptions_w2v2_chunk, wer_file = wer_chunk(results, words)
            ref_transcriptions = clean_french_disfluencies_repetition(ref_transcriptions)
            corpus_wer["wer_stats"].append(ids=[wav_file],predict=[normalization(pred_transcriptions_w2v2_chunk)],target=[normalization(ref_transcriptions)])
            
            #style = wav_to_style[wav_file]
            
            """style_wers[style]["wer_stats"].append(
                ids=[wav_file],
                predict=[normalization(pred_transcriptions_w2v2_chunk)],
                target=[normalization(ref_transcriptions)]
            )"""
            WER = wer_segment(wav_file, ref_transcriptions, pred_transcriptions_w2v2_chunk)
            
            writer.writerow({"filename": wav_file,**info, "trans_w2vec_vad_chunk": " ".join(normalization(pred_transcriptions_w2v2_chunk)), "WER_w2vec_vad_chunk": WER,})
stats = corpus_wer["wer_stats"].summarize()

print("Corpus WER:", stats["WER"])
print("S:", stats["substitutions"])
print("D:", stats["deletions"])
print("I:", stats["insertions"])
print("N:", stats["num_scored_tokens"])
"""print("\n========== WER PAR STYLE ==========")

for style, wer_obj in style_wers.items():

    stats = wer_obj["wer_stats"].summarize()

    print(
        f"{style:20s} "
        f"WER={stats['WER']:.2f} "
        f"S={stats['substitutions']} "
        f"D={stats['deletions']} "
        f"I={stats['insertions']} "
        f"N={stats['num_scored_tokens']}"
    )"""

/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-CEREB/CCM-002710-01_L01.wav
/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-CEREB/CCM-003094-01_L01.wav


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


23.068124999999995


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-CEREB/CCM-003110-01_L01.wav
17.76937500000001


...

Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-CEREB/CCM-004523-01_L01.wav
6.3956249999999955


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`


/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-CEREB/CCM-004538-01_L01.wav
23.354999999999997
/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/8-CEREB/CCM-004773-01_L01.wav
Corpus WER: 28.142589118198874
S: 181
D: 69
I: 50
N: 1066


'print("\n========== WER PAR STYLE ==========")\n\nfor style, wer_obj in style_wers.items():\n\n    stats = wer_obj["wer_stats"].summarize()\n\n    print(\n        f"{style:20s} "\n        f"WER={stats[\'WER\']:.2f} "\n        f"S={stats[\'substitutions\']} "\n        f"D={stats[\'deletions\']} "\n        f"I={stats[\'insertions\']} "\n        f"N={stats[\'num_scored_tokens\']}"\n    )'

In [42]:
#rhapsodie
"""import pandas as pd
df=pd.read_csv(csv_path)[["filename","trans_w2vec_vad_chunk"]]
df["speaker"] = [i.split("/")[-1].split(".")[0] for i in df["filename"]]
df["speaker"] = [i.split("-")[1]+"-"+i.split("-")[0] for i in df["speaker"]]
df = df.rename(columns={"trans_w2vec_vad_chunk": "transcription"})"""

'import pandas as pd\ndf=pd.read_csv(csv_path)[["filename","trans_w2vec_vad_chunk"]]\ndf["speaker"] = [i.split("/")[-1].split(".")[0] for i in df["filename"]]\ndf["speaker"] = [i.split("-")[1]+"-"+i.split("-")[0] for i in df["speaker"]]\ndf = df.rename(columns={"trans_w2vec_vad_chunk": "transcription"})'

In [44]:
"""df=pd.read_csv(csv_path)[["filename","trans_w2vec_vad_chunk"]]
df["speaker"] = ["_".join(str(i).split("/")[-1].split("_")[:3]) for i in df["filename"]]
df = df.rename(columns={"trans_w2vec_vad_chunk": "transcription"})"""

In [104]:
df=pd.read_csv(csv_path)[["filename","trans_w2vec_vad_chunk"]]
df["speaker"]=[i.split("/")[-1].split(".")[0] for i in df["filename"]]
df = df.rename(columns={"trans_w2vec_vad_chunk": "transcription"})

In [105]:
import os
import shutil
import re

mfa_root = "/vol/experiments3/imbenamor/TAPAS-FRAIS/data/ASR_ctrl_MFA"

os.makedirs(mfa_root, exist_ok=True)

for _, row in df.iterrows():

    speaker = str(row["speaker"])
    wav_name = row["filename"]
    transcription = row["transcription"]

    speaker_dir = os.path.join(mfa_root, speaker)
    os.makedirs(speaker_dir, exist_ok=True)
    # WAV source
    

    # WAV destination
    
    dst_wav = speaker_dir+"/"+ wav_name.split("/")[-1]
    print(wav_name,dst_wav)
    shutil.copy2(wav_name, dst_wav)

    # LAB file
    base = os.path.splitext(wav_name)[0].split("/")[-1]

    lab_path = os.path.join(
        speaker_dir,
        f"{base}.lab"
    )

    with open(lab_path, "w", encoding="utf-8") as f:
        f.write(transcription + "\n")

/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/12-CTRL/AEX-CAB000-02_L01.wav /vol/experiments3/imbenamor/TAPAS-FRAIS/data/ASR_ctrl_MFA/AEX-CAB000-02_L01/AEX-CAB000-02_L01.wav
/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/12-CTRL/AEX-CAC000-02_L01.wav /vol/experiments3/imbenamor/TAPAS-FRAIS/data/ASR_ctrl_MFA/AEX-CAC000-02_L01/AEX-CAC000-02_L01.wav
/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/12-CTRL/AEX-CAG000-01_L01.wav /vol/experiments3/imbenamor/TAPAS-FRAIS/data/ASR_ctrl_MFA/AEX-CAG000-01_L01/AEX-CAG000-01_L01.wav
/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/12-CTRL/AEX-CLJ000-02_L01.wav /vol/experiments3/imbenamor/TAPAS-FRAIS/data/ASR_ctrl_MFA/AEX-CLJ000-02_L01/AEX-CLJ000-02_L01.wav
/vol/corpora/TAPAS_FRAIS/Data_partagees_ParisTypaloc-TapasFrais/12-CTRL/AEX-CML000-01_L01.wav /vol/experiments3/imbenamor/TAPAS-FRAIS/data/ASR_ctrl_MFA/AEX-CML000-01_L01/AEX-CML000-01_L01.wav
/vol/corpora/TAPAS_FRAIS/Data_partagees_